# Reproducing Figure 1 of Guo et al. (2017), *On Calibration of Modern Neural Networks*

Figure 1 shows confidence histograms (top) and reliability diagrams (bottom) for a 5-layer LeNet and a 110-layer ResNet trained on CIFAR-100, **without** any post-hoc calibration.

**Runtime:** designed for Colab with a GPU. LeNet trains in a few minutes. ResNet-110 (64k iterations) takes on the order of 1–3 h depending on the GPU (an estimate, not measured). Checkpoints are saved every epoch, so training resumes after a disconnect (set `USE_DRIVE=True` so they survive a runtime reset).

### Assumptions (the paper does not pin these down)
1. **LeNet architecture & training.** The paper only says "LeNet 5" (LeCun et al., 1998). I use the classic layout (conv6-5×5 → pool → conv16-5×5 → pool → fc120 → fc84 → fc100) with ReLU and max-pooling, and SGD (lr 0.01, momentum 0.9, weight decay 5e-4, 120 epochs, lr ÷10 at epochs 60 and 90). All of these are my choices.
2. **ResNet-110 training** follows He et al. (2016), since Guo et al. say they use each architecture's original recipe: SGD, momentum 0.9, weight decay 1e-4, batch 128, lr 0.1 ÷10 at 32k and 48k iterations, stop at 64k; option-A (zero-padding) shortcuts; per-pixel mean subtraction; 4-px pad + random crop + horizontal flip; lr 0.01 warm-up for the first 400 iterations (He et al. describe this for ResNet-110 on CIFAR-10).
3. **Split:** 45,000 / 5,000 / 10,000 train/val/test (Section 5 of the paper). The validation set is not used for Figure 1.
4. **Bins:** the histogram looks like 20 bins and the reliability diagram like 10 bins, read off the figure by eye. The red "Gap" bar is drawn from the bin accuracy up to the bin **midpoint** (the diagonal), which is what the figure appears to show. Set `GAP_TARGET="mean_conf"` to use the mean confidence per bin instead.
5. **Mixed precision** (`USE_AMP=True`) is a speed-up that was not in the original setup. Set it to `False` for maximum fidelity.

### Numbers to compare against (from the paper itself)
| | Test error | ECE (M=15) |
|---|---|---|
| LeNet 5 | 44.92 % (Table S2); 44.9 in Fig. 1 | 4.85 % (Table 1) |
| ResNet 110 | 27.83 % (Table S2); **30.6 in Fig. 1** | 16.53 % (Table 1) |

The paper's own ResNet error in Fig. 1 (30.6) doesn't match Table S2 (27.83), so Fig. 1 may come from a different run. Expect your numbers to vary run-to-run as well.

In [ ]:
import os, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

SEED       = 0
QUICK_TEST = False   # True: tiny smoke test (a few iterations, 2k samples) to check the pipeline end to end
USE_AMP    = True    # mixed precision on GPU (not in the original paper)
USE_DRIVE  = False   # True: keep checkpoints/logits on Google Drive so they survive runtime resets
GAP_TARGET = "midpoint"  # or "mean_conf"

OUT_DIR = "./guo17_fig1"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/guo17_fig1"
if QUICK_TEST:
    OUT_DIR += "_quicktest"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = USE_AMP and device == "cuda"
torch.manual_seed(SEED); np.random.seed(SEED)
print("torch", torch.__version__, "| device:", device, "| AMP:", USE_AMP, "| out:", OUT_DIR)

## Data: CIFAR-100, 45k/5k/10k split, He et al. (2016) preprocessing

In [ ]:
DATA_DIR = "./data"
raw_train = torchvision.datasets.CIFAR100(DATA_DIR, train=True,  download=True)
raw_test  = torchvision.datasets.CIFAR100(DATA_DIR, train=False, download=True)

perm = np.random.RandomState(SEED).permutation(len(raw_train))
train_idx, val_idx = perm[:45000], perm[45000:]
test_idx = np.arange(len(raw_test))
if QUICK_TEST:
    train_idx, test_idx = train_idx[:2000], test_idx[:2000]

# Per-pixel mean image computed on the 45k training images only
mean_img = torch.tensor(raw_train.data[perm[:45000]].mean(axis=0) / 255.0,
                        dtype=torch.float32).permute(2, 0, 1)

class SubtractMean:
    def __init__(self, m): self.m = m
    def __call__(self, x): return x - self.m

train_tf = T.Compose([T.ToTensor(), SubtractMean(mean_img),
                      T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
test_tf  = T.Compose([T.ToTensor(), SubtractMean(mean_img)])

train_set = Subset(torchvision.datasets.CIFAR100(DATA_DIR, train=True,  transform=train_tf), train_idx)
test_set  = Subset(torchvision.datasets.CIFAR100(DATA_DIR, train=False, transform=test_tf),  test_idx)

BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=500,   shuffle=False, num_workers=2, pin_memory=True)
ITERS_PER_EPOCH = len(train_loader)
print(len(train_set), "train |", len(test_set), "test |", ITERS_PER_EPOCH, "iters/epoch")

## Models

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5), nn.ReLU(), nn.MaxPool2d(2),     # 32 -> 28 -> 14
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2))    # 14 -> 10 -> 5
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))


class ShortcutA(nn.Module):
    # Option A of He et al.: identity with spatial subsampling and zero-padded channels
    def __init__(self, in_planes, out_planes, stride):
        super().__init__()
        self.stride, self.pad = stride, out_planes - in_planes
    def forward(self, x):
        x = x[:, :, ::self.stride, ::self.stride]
        return F.pad(x, (0, 0, 0, 0, self.pad // 2, self.pad - self.pad // 2))


class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.shortcut = (ShortcutA(in_planes, planes, stride)
                         if stride != 1 or in_planes != planes else nn.Identity())
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetCifar(nn.Module):
    # depth = 6n + 2; ResNet-110 -> n = 18 blocks per stage, widths 16/32/64
    def __init__(self, depth=110, num_classes=100):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.conv = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(16)
        layers, in_planes = [], 16
        for planes, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(BasicBlock(in_planes, planes, stride if i == 0 else 1))
                in_planes = planes
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(64, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.layers(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print("LeNet5 params:", n_params(LeNet5()))
print("ResNet-110 params:", n_params(ResNetCifar(110)), "(He et al. report ~1.7M for the CIFAR-10 version)")

## Training (resumable)

In [ ]:
ipe = ITERS_PER_EPOCH
CONFIGS = {
    "lenet5": dict(builder=lambda: LeNet5(100), lr=0.01, momentum=0.9, wd=5e-4,
                   total_iters=120 * ipe, milestones=[60 * ipe, 90 * ipe],
                   warmup_iters=0, warmup_factor=1.0),
    "resnet110": dict(builder=lambda: ResNetCifar(110, 100), lr=0.1, momentum=0.9, wd=1e-4,
                      total_iters=64000, milestones=[32000, 48000],
                      warmup_iters=400, warmup_factor=0.1),
}
if QUICK_TEST:
    for c in CONFIGS.values():
        c.update(total_iters=30, milestones=[15, 25], warmup_iters=min(c["warmup_iters"], 5))

def lr_factor(it, cfg):
    if it < cfg["warmup_iters"]:
        return cfg["warmup_factor"]
    return 0.1 ** sum(it >= m for m in cfg["milestones"])

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):   # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)

@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    zs, ys = [], []
    for x, y in loader:
        zs.append(model(x.to(device, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy()

def train_model(name, cfg):
    ckpt_path = os.path.join(OUT_DIR, f"{name}_ckpt.pt")
    model = cfg["builder"]().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=cfg["momentum"],
                          weight_decay=cfg["wd"])
    scaler = make_scaler(USE_AMP)
    it = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"]); it = ck["it"]
        print(f"[{name}] resumed at iteration {it}")
    total = cfg["total_iters"]
    while it < total:
        model.train(); t0, loss_sum, nb = time.time(), 0.0, 0
        for x, y in train_loader:
            if it >= total:
                break
            for g in opt.param_groups:
                g["lr"] = cfg["lr"] * lr_factor(it, cfg)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            it += 1; loss_sum += loss.item(); nb += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                        scaler=scaler.state_dict(), it=it), ckpt_path)
        epoch = (it + ipe - 1) // ipe
        msg = f"[{name}] epoch {epoch:3d} it {it:6d}/{total} loss {loss_sum/max(nb,1):.3f} lr {opt.param_groups[0]['lr']:.4f} ({time.time()-t0:.0f}s)"
        if epoch % 10 == 0 or it >= total:
            z, y_ = get_logits(model, test_loader)
            msg += f" | test err {100*(z.argmax(1) != y_).mean():.2f}%"
        print(msg)
    return model

LOGITS = {}
for name, cfg in CONFIGS.items():
    path = os.path.join(OUT_DIR, f"{name}_test_logits.npz")
    if not os.path.exists(path):
        model = train_model(name, cfg)
        z, y = get_logits(model, test_loader)   # final model, no post-hoc calibration
        np.savez(path, logits=z, labels=y)
        del model; torch.cuda.empty_cache() if device == "cuda" else None
    d = np.load(path)
    LOGITS[name] = (d["logits"], d["labels"])
    print(name, "test logits:", d["logits"].shape)

## Metrics and plotting (NumPy / matplotlib only)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def confidence_and_correct(logits, labels):
    p = softmax_np(logits.astype(np.float64))
    return p.max(axis=1), (p.argmax(axis=1) == labels).astype(float)

def bin_index(conf, M):
    # bins I_m = ((m-1)/M, m/M], as in Section 2 of the paper
    return np.clip(np.ceil(conf * M).astype(int) - 1, 0, M - 1)

def ece_score(conf, correct, M=15):
    b, n, e = bin_index(conf, M), len(conf), 0.0
    for m in range(M):
        mask = b == m
        if mask.any():
            e += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())
    return e

def plot_figure1(panels, hist_bins=20, rel_bins=10, gap_target="midpoint", save_path=None):
    # panels: dict {title: (conf, correct)}
    plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm"})
    k = len(panels)
    fig, axes = plt.subplots(2, k, figsize=(3.3 * k, 6.4), sharex=True, squeeze=False)
    edges = np.linspace(0, 1, rel_bins + 1); w = 1.0 / rel_bins; mids = edges[:-1] + w / 2
    for j, (title, (conf, correct)) in enumerate(panels.items()):
        acc, avg = correct.mean(), conf.mean()
        # --- top: confidence histogram ---
        ax = axes[0, j]
        ax.hist(conf, bins=np.linspace(0, 1, hist_bins + 1),
                weights=np.full(len(conf), 1.0 / len(conf)),
                color="blue", edgecolor="black", linewidth=0.6)
        lo, hi = sorted([(avg, "Avg. confidence"), (acc, "Accuracy")])
        for (x, lab), side in [(lo, -1), (hi, 1)]:
            ax.axvline(x, ls="--", lw=2, color="gray")
            ax.text(x + side * 0.015, 0.55, lab, rotation=90, ha="right" if side < 0 else "left",
                    va="center", fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.set_ylim(0, 1)
        # --- bottom: reliability diagram ---
        ax = axes[1, j]
        b = bin_index(conf, rel_bins)
        nonempty = np.array([(b == m).any() for m in range(rel_bins)])
        accs = np.array([correct[b == m].mean() if nonempty[m] else 0.0 for m in range(rel_bins)])
        target = mids if gap_target == "midpoint" else np.array(
            [conf[b == m].mean() if nonempty[m] else mids[m] for m in range(rel_bins)])
        ax.bar(edges[:-1], accs, width=w, align="edge", color="blue",
               edgecolor="black", linewidth=0.6, label="Outputs")
        ax.bar(edges[:-1][nonempty], np.abs(target - accs)[nonempty],
               bottom=np.minimum(target, accs)[nonempty], width=w, align="edge",
               color="red", alpha=0.3, edgecolor="red", hatch="//", linewidth=0.8, label="Gap")
        ax.plot([0, 1], [0, 1], ls="--", color="gray", lw=1.5)
        ax.text(0.97, 0.03, f"Error={100 * (1 - acc):.1f}", transform=ax.transAxes,
                ha="right", va="bottom", fontsize=12,
                bbox=dict(facecolor="lightsteelblue", edgecolor="black", alpha=0.9))
        ax.legend(loc="upper left", fontsize=9)
        ax.set_ylim(0, 1)
        for a in axes[:, j]:
            a.set_xlim(0, 1); a.set_facecolor("#f7f7f7")
            a.grid(True, ls=":", color="gray", alpha=0.7); a.set_axisbelow(True)
            a.set_xticks(np.linspace(0, 1, 6))
        if j > 0:
            axes[0, j].set_yticklabels([]); axes[1, j].set_yticklabels([])
    axes[0, 0].set_ylabel("% of Samples"); axes[1, 0].set_ylabel("Accuracy")
    fig.supxlabel("Confidence", y=0.02)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    return fig

## Figure 1

In [ ]:
TITLES = {"lenet5": "LeNet (1998)\nCIFAR-100", "resnet110": "ResNet (2016)\nCIFAR-100"}
PAPER  = {"lenet5": (44.92, 4.85), "resnet110": (27.83, 16.53)}   # Table S2 error, Table 1 ECE

panels = {}
for name, (z, y) in LOGITS.items():
    conf, correct = confidence_and_correct(z, y)
    panels[TITLES[name]] = (conf, correct)
    err, ece = 100 * (1 - correct.mean()), 100 * ece_score(conf, correct, M=15)
    print(f"{name:10s} error {err:6.2f}% (paper {PAPER[name][0]}%) | "
          f"ECE(M=15) {ece:5.2f}% (paper {PAPER[name][1]}%) | "
          f"avg conf {conf.mean():.3f} vs acc {correct.mean():.3f}")

fig = plot_figure1(panels, gap_target=GAP_TARGET, save_path=os.path.join(OUT_DIR, "figure1.png"))
plt.show()